In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import pandas as pd
import numpy as np
DATA = pd.read_csv('sign_mnist_train.csv')
DATA_TEST = pd.read_csv('sign_mnist_test.csv')
DATA

FileNotFoundError: [Errno 2] No such file or directory: 'sign_mnist_train.csv'

In [ ]:
class CNN(nn.Module):
    def __init__(self, classes):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 3, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(8*7*7, 128)
        self.fc_ = nn.Linear(128, classes)

    def forward(self, x):
        # x = x[:, 1:783] # Only take the first 784 elements if there's an extra column
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 8*7*7)
        x = torch.relu(self.fc(x))
        x = self.fc_(x)
        return x

In [ ]:
transform = transforms.ToTensor()

class CustomDataset(Dataset):
    def __init__(self, dataframe:pd.DataFrame, transform=None):
        """
        Initializes the dataset.
        Args:
            dataframe (pd.DataFrame): CSV Dataframe from pandas
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.labels = dataframe['label'].values
        self.images = dataframe.drop('label', axis=1).values.reshape(-1, 1, 28, 28)
        self.transform = transform

    def __len__(self):
        """
        Returns the total number of samples in the dataset.
        """
        return len(self.labels)

    def __getitem__(self, idx):
        """
        Retrieves a single sample from the dataset at the given index.
        Args:
            idx (int): Index of the sample to retrieve.
        Returns:
            tuple: (features, label) for the sample.
        """

        features = torch.tensor(self.images[idx], dtype=torch.float32) / 255.0
        label = torch.tensor(self.labels[idx], dtype=torch.long) # Or float32 for regression

        if self.transform:
            features = self.transform(features)

        return features, label

train_set = CustomDataset(DATA)
test_set = CustomDataset(DATA_TEST)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64)
for images, labels in train_loader:
    print(images)
    print()
    print(labels)
    break

tensor([[[[0.5373, 0.5451, 0.5569,  ..., 0.6196, 0.6157, 0.6118],
          [0.5373, 0.5451, 0.5569,  ..., 0.6196, 0.6196, 0.6196],
          [0.5412, 0.5490, 0.5608,  ..., 0.6157, 0.6196, 0.6157],
          ...,
          [0.5294, 0.5373, 0.5451,  ..., 0.7059, 0.6392, 0.6824],
          [0.5451, 0.5529, 0.5569,  ..., 0.6431, 0.6627, 0.6157],
          [0.4314, 0.4353, 0.4588,  ..., 0.4745, 0.6706, 0.6392]]],


        [[[0.4157, 0.4235, 0.4392,  ..., 0.5451, 0.5373, 0.5333],
          [0.4235, 0.4353, 0.4431,  ..., 0.5529, 0.5490, 0.5412],
          [0.4275, 0.4353, 0.4471,  ..., 0.5608, 0.5529, 0.5451],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.6863, 0.6824, 0.6706],
          [0.0000, 0.0118, 0.0000,  ..., 0.6863, 0.6784, 0.6706],
          [0.0000, 0.0510, 0.0039,  ..., 0.6902, 0.6706, 0.6667]]],


        [[[0.8039, 0.8196, 0.8275,  ..., 0.8118, 0.8039, 0.8000],
          [0.8196, 0.8275, 0.8392,  ..., 0.8275, 0.8196, 0.8118],
          [0.8275, 0.8392, 0.8549,  ..

In [ ]:
model = CNN(26)
optimizer = optim.SGD(model.parameters(), lr = 0.01, momentum=0.9)
# optimizer = optim.Adam(model.parameters(), lr = 0.001) # Use either one
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# Training the Model
epochs = 3
for i in range(epochs):
    total, correct = 0, 0
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f"Epoch {i + 1}: \n\t Accuracy: {correct/total: .4f} \n")


Epoch 1: 
	 Accuracy:  0.1096 

Epoch 2: 
	 Accuracy:  0.5703 

Epoch 3: 
	 Accuracy:  0.8040 

